In [1]:
import os
import amigo

In [2]:
uncal_path = "/Users/mcha5804/JWST/FLATS/uncal/"
data_path = "/Users/mcha5804/JWST/FLATS/calslope/"

# Get the list of files in the directory
files = os.listdir(data_path)
files = files[0:2]

In [3]:
import os
import shutil
import jax.numpy as np
from astropy.io import fits
from amigo.misc import tqdm
from amigo.pipelines import *


def process_calslope(
    directory,
    output_dir="calslope/",
    sigma=5,
    chunk_size=0,
    n_groups=None,  # how many groups of the ramp to use
    clean_dir=True,
):
    """
    Chunk size determines the maximum number of integrations in a 'chunk'. Each chunk
    is saved to its own file with an integer extension added. This breaks the data set
    into smaller time series to help avoid issues with any time-variation in the data.
    A chunk_size of zero will do no chunking and process the data all in one.

    This will (presently) always reprocess data

    if clean_dir is True, the existisng contents of the output_dir will be deleted prior
    to procesing, so ensure no old files are hanging around
    """
    if directory[-1] != "/":
        directory += "/"
    if output_dir[-1] != "/":
        output_dir += "/"

    # Get the files
    files = [directory + f for f in os.listdir(directory) if f.endswith("_uncal.fits")]

    # Check if there are any files to process
    if len(files) == 0:
        print("No _ramp.fits files found, no processing done.")
        return

    # Get the file paths
    paths = files[0].split("/")
    base_path = "/".join(paths[:-2]) + "/"
    output_path = base_path + output_dir

    # Check whether the specified output directory exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Clear the existing files (since we might use different chunk sizes, and we do not
    # want to have old files hang around)
    if clean_dir:
        print("Cleaning existing directory")
        delete_contents(output_path)

    # Iterate over files
    print("Running calslope processing...")
    for file_path in tqdm(files):

        file = fits.open(file_path)

        # Check if the file is a NIS_AMI file
        if file[0].header["EXP_TYPE"] not in ["NIS_AMI", "NIS_LAMP", "NIS_IMAGE"]:
            print("Not a supported file type, skipping...")
            continue

        # Skip single group files
        if file[0].header["NGROUPS"] == 1:
            print("Only one group, skipping...")
            continue

        # Get the data
        data = np.array(file["SCI"].data, int)

        # Copying header information
        sci_header = file["SCI"].header.copy(strip=True)
        sci_header.remove("EXTNAME")

        file.close()

        if chunk_size == 0:
            chunks = [data]
            nchunks = 1
        else:
            nints = data.shape[0]
            if nints < chunk_size:
                nchunks = 1
            else:
                nchunks = np.round(nints / chunk_size).astype(int)
            chunks = np.array_split(data, nchunks)
            print(f"Breaking into {nchunks} chunks")

        # Get the root of the file name
        file_name = file_path.split("/")[-1]
        file_root = "_".join(file_name.split("_")[:-2])

        for i, chunk in enumerate(chunks):

            # Check if the file is a NIS_AMI file
            file_name = file_root + f"_{i+1:0{4}}" + "_nis_calslope.fits"
            file_calslope = os.path.join(output_path + file_name)

            # Create the new file
            shutil.copy(file_path, file_calslope)

            # Open new file
            # file = fits.open(file_calslope, mode="update")
            file = fits.open(file_calslope)

            # Remove the redundant or undesired extensions
            del file["SCI"]
            del file["GROUP"]
            del file["INT_TIMES"]

            # Update the various headers
            file[0].header["NCHUNKS"] = int(nchunks)
            file[0].header["CHUNK"] = i + 1
            file[0].header["CHUNKSZ"] = int(chunk_size)
            file[0].header["NINTS"] = int(chunk.shape[0])
            file[0].header["FILENAME"] = file_name
            file[0].header["SIGMA"] = sigma
            file[0].header.extend(sci_header)  # Copy the science header

            # Process the chunk
            file = process_data(file, chunk, sigma=sigma)

            # Save as calslope
            file.writeto(file_calslope, overwrite=True)
            file.close()

    print("Done\n")
    return output_path

In [4]:
process_calslope(uncal_path)

Cleaning existing directory
Running calslope processing...


  0%|          | 0/177 [00:00<?, ?it/s]

OSError: Not enough space on disk: requested 1962967107, available 370692096. 83886080 requested and 0 written

In [ ]:
# Bind file path, type and exposure type
file_fn = lambda data_path, **kwargs: amigo.files.get_files(
    data_path,
    "calslope",
    **kwargs,
)

files = file_fn(data_path)

exps = amigo.model_fits.Exposure(files[0])

KeyError: "Keyword 'IS_PSF' not found."

In [ ]:
files[0][0].header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    0 / number of array dimensions                     
EXTEND  =                    T                                                  
DATE    = '2024-12-13T06:22:50.891' / UTC date file created                     
ORIGIN  = 'STSCI   '           / Organization responsible for creating file     
TIMESYS = 'UTC     '           / principal time system for time-related keywords
TIMEUNIT= 's       '           / Default unit applicable to all time values     
FILENAME= 'jw04472040001_01201_00001_0001_nis_calslope.fits' / Name of the file 
SDP_VER = '2024_3a '           / Data processing (DP) Software Version          
PRD_VER = 'PRDOPSSOC-068'      / S&OC Project Reference Database (PRD) Version  
OSS_VER = '9.2     '           / Observatory Scheduling Software (OSS) Version  
GSC_VER = 'GSC31   '        